# 02 — MZA-FAO simulation of second-season maize (399 municipalities × 27 years × 9 dates)

Runs `modelo_mza.simular` (crop water balance, PPf and PA from the **agrometeorologiapy 0.3.0** library)
for each municipality of Paraná, sowing year 1999–2025 and nine sowing dates (days 5, 15 and 25 of
January to March), with the BR-DWGD daily weather of notebook 01.

Model choices (see `modelo_mza.py`):

- **Growing-degree-day phenology** (Tb = 10 °C, 1450 °C day to physiological maturity): late sowing
  lengthens the cycle and pushes grain filling into winter.
- **Kc** in the four FAO-56 stages (0.3 / 1.2 / 0.6) applied to the BR-DWGD Penman-Monteith ETo.
- **ky** in the five periods of Doorenbos & Kassam (1979): establishment 0.4; vegetative 0.4;
  flowering 1.5; grain filling 0.5; maturity 0.2 — PA by the product over phases.
- **Roots** from 0.10 m to 0.40 m until the end of the vegetative phase; CAD = z × municipal DTA.
- 30 days of water balance before sowing (straw, initial Kc) for the initial soil water.
- **ZARC criteria** (Portaria SPA/MAPA nº 329/2026) on a 100-day reference cycle, with phases on the
  fixed days of the ordinance, for the official risk classification.

Takes ~25 min on 4 cores.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import pandas as pd

import modelo_mza as M
from sensibilidade import PARAM, SAIDA

RAIZ = Path.cwd().parent
DADOS, RESULTADOS = RAIZ / "dados", RAIZ / "resultados"

ANOS = range(1999, 2026)
DATAS_SEMEADURA = [(m, d) for m in (1, 2, 3) for d in (5, 15, 25)]
CT = PARAM.get("ct", M.CT)  # thermal time (°C day); changes only in the sensitivity scenarios

municipios = pd.read_csv(DADOS / "municipios_pr.csv")
print(len(municipios), "municipalities ×", len(ANOS), "years ×", len(DATAS_SEMEADURA), "dates =",
      len(municipios) * len(ANOS) * len(DATAS_SEMEADURA), "simulations")

399 municipalities × 27 years × 9 dates = 96957 simulations


In [2]:
# per-municipality reading: a filterable parquet avoids loading 4 million rows in each process
if not (DADOS / "xavier_diario_pr.parquet").exists():
    pd.read_csv(DADOS / "xavier_diario_pr.csv.gz", parse_dates=["data"]).to_parquet(
        DADOS / "xavier_diario_pr.parquet", index=False)

arquivo = DADOS / "xavier_diario_pr.parquet"
with ProcessPoolExecutor(max_workers=4) as ex:
    futuros = [ex.submit(M.simular_municipio, r.cod_ibge, r.latitude, r.dta_mm_m, arquivo, ANOS, DATAS_SEMEADURA, CT)
               for r in municipios.itertuples()]
    sim = pd.concat([f.result() for f in futuros], ignore_index=True)

sim.to_parquet(SAIDA / "simulacao_mza_pr.parquet", index=False)
print(sim.shape)

(96957, 70)


## Overview

In [3]:
resumo = sim.groupby("semeadura")[["ciclo_dias", "PPf", "PA", "ISNA_floracao", "ISNA_enchimento",
                                   "geada_floracao", "geada_enchimento"]].mean()
resumo.loc[[f"{d:02d}/{m:02d}" for m, d in DATAS_SEMEADURA]].round(2)

,ciclo_dias,PPf,PA,ISNA_floracao,ISNA_enchimento,geada_floracao,geada_enchimento
semeadura,,,,,,,
05/01,105.20,6931.80,2932.68,0.73,0.66,0.00,0.04
15/01,109.69,6826.33,2535.22,0.70,0.63,0.00,0.12
25/01,115.79,6703.26,2203.11,0.66,0.61,0.00,0.29
05/02,124.13,6570.27,1917.13,0.63,0.60,0.01,0.55
15/02,132.58,6438.66,1766.20,0.59,0.60,0.05,0.83
25/02,140.59,6348.50,1738.20,0.58,0.62,0.15,1.07
05/03,146.81,6290.37,1811.18,0.60,0.64,0.26,1.15
15/03,153.46,6257.69,1838.43,0.61,0.66,0.40,1.15
25/03,158.49,6272.38,1945.31,0.66,0.63,0.54,0.98
